In [18]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [19]:
from langchain_groq import ChatGroq

llm=ChatGroq(model="llama-3.3-70b-versatile")

In [20]:
from typing import Annotated
from langchain_tavily import TavilySearch
from langchain_core.tools import tool
from typing_extensions import TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

from langgraph.types import Command, interrupt

class State(TypedDict):
    messages:Annotated[list, add_messages]

graph_builder = StateGraph(State)

@tool
def human_assistance(query : str) -> str:
    """Request assistance from a human"""
    human_response = interrupt({"query":query})
    return human_response["data"]

tool = TavilySearch(max_results=2)
tools =[tool, human_assistance]
llm_with_tools = llm.bind_tools(tools)

def chatbot(state: State):
    message = llm_with_tools.invoke(state["messages"])

    return {"messages": [message]}

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools))
graph_builder.add_conditional_edges("chatbot", tools_condition)

graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

In [21]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()

In [24]:
user_input = (
    "I need some expert guidance and assistance for building an AI agent. "
    "Could you request assistance for me?"
)

config = {
    "configurable": {
        "thread_id": "1"
    }
}

graph = graph_builder.compile(checkpointer=memory)

events = graph.stream(
    {"messages": user_input},
    config=config,
    stream_mode="values"
)

for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================ Human Message =================================

I need some expert guidance and assistance for building an AI agent. Could you request assistance for me?
================================== Ai Message ==================================
Tool Calls:
  human_assistance (echwaecj2)
 Call ID: echwaecj2
  Args:
    query: I need expert guidance and assistance for building an AI agent.
================================== Ai Message ==================================
Tool Calls:
  human_assistance (echwaecj2)
 Call ID: echwaecj2
  Args:
    query: I need expert guidance and assistance for building an AI agent.


In [27]:
human_response = (
    "We, the experts are here to help, We'd recommend you check out langraph to build your agent. It is much more reliable and extensible that simple autonomous agents."

)

human_command = Command(resume={"data":human_response})

events = graph.stream(human_command, config, stream_mode="values")
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================== Ai Message ==================================
Tool Calls:
  human_assistance (2d79fhhp2)
 Call ID: 2d79fhhp2
  Args:
    query: Explain how to use langraph for building an AI agent.
================================= Tool Message =================================
Name: human_assistance

We, the experts are here to help, We'd recommend you check out langraph to build your agent. It is much more reliable and extensible that simple autonomous agents.
================================== Ai Message ==================================
Tool Calls:
  human_assistance (k3ma04qtb)
 Call ID: k3ma04qtb
  Args:
    query: Explain how to use langraph for building an AI agent and provide resources for learning more about langraph.
================================== Ai Message ==================================
Tool Calls:
  human_assistance (k3ma04qtb)
 Call ID: k3ma04qtb
  Args:
    query: Explain how to use langraph for building an AI agent and provide resources for 